In [ ]:
# ===== C1 clone + clock =====
import time, subprocess
NB_START=time.time()
subprocess.run("mkdir -p /kaggle/temp && cd /kaggle/temp && rm -rf FreeFine && git clone -q https://github.com/CIawevy/FreeFine.git", shell=True, check=True); print("cloned")

In [ ]:
%%bash
# ===== C2 freefine_env (generation) =====
set -e
pip install -q --root-user-action=ignore uv; uv python install 3.10.13
V=/kaggle/temp/freefine_env; PY=$V/bin/python
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" "torch==2.1.1" "torchvision==0.16.1" --index-url https://download.pytorch.org/whl/cu121
cd /kaggle/temp/FreeFine
uv pip install --python "$PY" -r requirements.txt || { grep -v '^xformers' requirements.txt>/tmp/r.txt; uv pip install --python "$PY" -r /tmp/r.txt; uv pip install --python "$PY" xformers; }
uv pip install --python "$PY" einops==0.7.0 omegaconf==2.3.0 "setuptools<70"
"$PY" -c "import torch,diffusers,xformers; print('freefine_env OK',torch.__version__)"

In [ ]:
%%bash
# ===== C3 metric_env (evaluation) =====
set -e
V=/kaggle/temp/metric_env; PY=$V/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf "$V"; uv venv --python 3.10.13 "$V"
uv pip install --python "$PY" torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python "$PY" "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/m.txt
uv pip install --python "$PY" -r /tmp/m.txt
uv pip install --python "$PY" "setuptools<70"
uv pip install --python "$PY" --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python "$PY" "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $V -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null||true; done
echo "metric_env OK"

In [ ]:
# ===== C4 patch metrics (args.3d, SD-2.1 mirror, SEEDED MD) + model.py START_LAYER fix =====
import pathlib, re
mr=pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
mp=mr/"main.py"; mp.write_text(mp.read_text().replace("args.3d","getattr(args,'3d')"))
for f in [mr/"MD"/"mean_distance.py", mr/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1","sd2-community/stable-diffusion-2-1"))
md=mr/"MD"/"mean_distance.py"; s=md.read_text()
s=s.replace("all_dist = []","all_dist = []\n    import torch as _st; _st.manual_seed(42); _st.cuda.manual_seed_all(42)",1); md.write_text(s)
mm=pathlib.Path("/kaggle/temp/FreeFine/src/demo/model.py"); g=mm.read_text()
if not re.search(r'^\s*import os\b', g, re.M): g="import os\n"+g
assert "list(range(10, 16))" in g, "layer_idx hardcode missing"
n=g.count("list(range(10, 16))")
g=g.replace("list(range(10, 16))","list(range(int(os.environ.get('FF_START_LAYER','10')), 16))")
mm.write_text(g); print(f"patched metrics + model.py start_layer ({n} sites)")

In [ ]:
# ===== C5 parametrize inference script =====
import os
P="/kaggle/temp/FreeFine/evaluation/FreeFine"; src=open(f"{P}/freefine_batch_infer_2d.py").read()
src=src.replace("sys.path.append('/data/Hszhu/FreeFine')","sys.path.append('/kaggle/temp/FreeFine')")
src=src.replace('pretrained_model_path = "/data/Hszhu/prompt-to-prompt/stable-diffusion-v1-5/"','pretrained_model_path = "stable-diffusion-v1-5/stable-diffusion-v1-5"')
old=('        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n        obj_label = ""\n        ori_mask = read_and_resize_mask(ori_mask_path)\n')
new=('        ori_mask = read_and_resize_mask(ori_mask_path)\n'
     '        coarse_input, target_mask = re_edit_2d(ori_img, ori_mask, edit_param, inp_back_ground)\n'
     '        obj_label = (case.get("obj_label","") if os.environ.get("FF_USE_PROMPT")=="1" else "")\n')
assert old in src, "ori_mask/obj_label block mismatch"; src=src.replace(old,new,1)
src=src.replace('"guidance_scale": 7.5,','"guidance_scale": float(os.environ.get("FF_GUIDANCE","7.5")),')
src=src.replace('"start_step": 35,','"start_step": int(os.environ.get("FF_START_STEP","35")),')
src=src.replace('dataset_json = osp.join(dst_base, "annotations_2d.json")','dataset_json = os.environ.get("FF_SUBSET_JSON", osp.join(dst_base,"annotations_2d.json"))')
src=src.replace('dst_gen_dir = osp.join(dst_base, "Geo-Bench-2D/Gen_results_FreeFine_2d")','dst_gen_dir = os.environ.get("FF_OUT_DIR", osp.join(dst_base,"Geo-Bench-2D/Gen_results_FreeFine_2d"))')
src=src.replace('base_dir = "/data/Hszhu/dataset/GeoBenchMeta/"','base_dir = "/kaggle/temp/GeoBenchMeta"')
open(f"{P}/freefine_sweep_2d.py","w").write(src)
assert all(x in src for x in ["FF_USE_PROMPT","FF_GUIDANCE","FF_START_STEP"]), "parametrize failed"
print("parametrized script written")

In [ ]:
# ===== C6 data + balanced-200 =====
import os, glob, json, csv, random, shutil
from collections import defaultdict, Counter
GEO="/kaggle/temp/GeoBenchMeta"; os.makedirs(f"{GEO}/Geo-Bench-2D",exist_ok=True)
CACHE=next(c for c in glob.glob("/kaggle/input/**/Geo-Bench-2D",recursive=True) if os.path.isdir(f"{c}/source_img"))
COARSE=glob.glob("/kaggle/input/**/coarse_img/*/*/*.png",recursive=True)[0].split("/coarse_img/")[0]+"/coarse_img"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
IB=os.path.dirname(os.path.dirname(os.path.dirname(glob.glob("/kaggle/input/**/inp_img_blended/**/inp_img.png",recursive=True)[0])))
ANNs=glob.glob("/kaggle/input/**/annotation_2d.json",recursive=True)[0]; META=glob.glob("/kaggle/input/**/sample_metadata.csv",recursive=True)[0]
for nm in ["source_img","source_mask","target_mask","source_img_full_v2"]:
    d=f"{GEO}/Geo-Bench-2D/{nm}";  os.path.exists(d) or os.symlink(f"{CACHE}/{nm}",d)
for nm,sc in [("coarse_img",COARSE),("inp_img_blended",IB)]:
    d=f"{GEO}/Geo-Bench-2D/{nm}";  os.path.exists(d) or os.symlink(sc,d)
shutil.copy(ANNs,f"{GEO}/annotation_2d.json"); ann=json.load(open(f"{GEO}/annotation_2d.json"))
meta=[r for r in csv.DictReader(open(META)) if os.path.exists(f"{IB}/{r['da_n']}/{r['ins_id']}/inp_img.png")]
random.seed(42); cells=defaultdict(list)
for r in meta: cells[(r["edit_type"],r["difficulty"])].append(r)
keys=sorted(cells); per=200//len(keys); picked=[]
for k in keys:
    pool=cells[k][:]; random.shuffle(pool); picked+=pool[:per]
ch={(r["da_n"],r["ins_id"],r["case_id"]) for r in picked}
left=[r for r in meta if (r["da_n"],r["ins_id"],r["case_id"]) not in ch]; random.shuffle(left)
for r in left:
    if len(picked)>=200: break
    picked.append(r)
picked=picked[:200]; json.dump(picked,open(f"{GEO}/subset_meta.json","w"))
print("subset",len(picked),dict(Counter(r["edit_type"] for r in picked)))
gsub={}
for r in picked:
    d,i,e=r["da_n"],r["ins_id"],r["case_id"]; lf=dict(ann[d]["instances"][i][e])
    lf["ori_img_path"]=os.path.join(GEO,lf["ori_img_path"]); lf["ori_mask_path"]=os.path.join(GEO,lf["ori_mask_path"])
    gsub.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
json.dump(gsub,open(f"{GEO}/gen_subset.json","w"))
os.makedirs(f"{GEO}/gen_eval",exist_ok=True)
d=f"{GEO}/gen_eval/baseline"
if os.path.islink(d): os.remove(d)
os.symlink(GENBASE,d); print("baseline eval set linked")

In [ ]:
# ===== C7 validation gate (default params must reproduce baseline) =====
import os, json, socket, subprocess, glob, numpy as np
from PIL import Image
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"]=UserSecretsClient().get_secret("HF_TOKEN")
GEO="/kaggle/temp/GeoBenchMeta"; P="/kaggle/temp/FreeFine/evaluation/FreeFine"
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
gs=json.load(open(f"{GEO}/gen_subset.json")); val={}; n=0
for d,da in gs.items():
    for i,ins in da["instances"].items():
        for e in ins:
            val.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=ins[e]; n+=1
            if n>=5:break
        if n>=5:break
    if n>=5:break
json.dump(val,open(f"{GEO}/val5.json","w")); os.makedirs("/kaggle/temp/val5",exist_ok=True)
env=os.environ.copy(); env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1","TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/val5.json","FF_OUT_DIR":"/kaggle/temp/val5"})
s=socket.socket();s.bind(("",0));port=s.getsockname()[1];s.close()
r=subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=1","--master-port",str(port),"freefine_sweep_2d.py"],cwd=P,env=env,capture_output=True,text=True)
imgs=sorted(glob.glob("/kaggle/temp/val5/**/*.png",recursive=True))
if not imgs: print((r.stdout+r.stderr)[-3000:]); raise SystemExit("validation 0 images")
diffs=[float(np.abs(np.array(Image.open(n).convert("RGB"),float)-np.array(Image.open(f"{GENBASE}/{os.path.relpath(n,'/kaggle/temp/val5')}").convert("RGB").resize(Image.open(n).size),float)).mean()) for n in imgs]
print("validation mean|Δ|:",[round(x,3) for x in diffs]); assert max(diffs)<1.0,"NOT REPRODUCING"; print("✓ validation passed")

Cell 8 — IP-A (%%writefile on line 1, then the full module — this is ip_ff.py with the geometry builders):

In [ ]:
%%writefile /kaggle/temp/FreeFine/ip_ff.py
import os, math, torch
import torch.nn as nn
import numpy as np
from PIL import Image

class ImageProjModel(nn.Module):
    def __init__(self, cross_attention_dim=768, clip_embeddings_dim=1024, clip_extra_context_tokens=4):
        super().__init__()
        self.clip_extra_context_tokens = clip_extra_context_tokens
        self.cross_attention_dim = cross_attention_dim
        self.proj = nn.Linear(clip_embeddings_dim, clip_extra_context_tokens * cross_attention_dim)
        self.norm = nn.LayerNorm(cross_attention_dim)
    def forward(self, image_embeds):
        x = self.proj(image_embeds).reshape(-1, self.clip_extra_context_tokens, self.cross_attention_dim)
        return self.norm(x)

def load_ip(model, ckpt_path, image_encoder_path, device, dtype=torch.float32, num_tokens=4,
            image_encoder_subfolder="models/image_encoder", enc_device="cpu"):
    from transformers import CLIPVisionModelWithProjection, CLIPImageProcessor
    sd = torch.load(ckpt_path, map_location="cpu")
    assert "image_proj" in sd and "ip_adapter" in sd, "unexpected IP-Adapter checkpoint format"
    image_proj = ImageProjModel(768, 1024, num_tokens)
    image_proj.load_state_dict(sd["image_proj"])
    image_proj = image_proj.to(enc_device, dtype=torch.float32).eval()
    attn2 = [(n, m) for n, m in model.unet.named_modules() if n.endswith("attn2")]
    ip = sd["ip_adapter"]
    prefixes = sorted({int(k.split(".")[0]) for k in ip}, key=int)
    assert len(attn2) == len(prefixes), f"cross-attn {len(attn2)} != IP {len(prefixes)}"
    for (name, mod), p in zip(attn2, prefixes):
        inner = mod.to_q.out_features
        kw = ip[f"{p}.to_k_ip.weight"]; vw = ip[f"{p}.to_v_ip.weight"]
        assert kw.shape == (inner, 768) and vw.shape == (inner, 768)
        k = nn.Linear(768, inner, bias=False); v = nn.Linear(768, inner, bias=False)
        k.weight.data.copy_(kw); v.weight.data.copy_(vw)
        mod.ip_to_k = k.to(device, dtype=dtype); mod.ip_to_v = v.to(device, dtype=dtype)
    enc = CLIPVisionModelWithProjection.from_pretrained(
        image_encoder_path, subfolder=image_encoder_subfolder).to(enc_device, dtype=torch.float32).eval()
    proc = CLIPImageProcessor()
    print(f"[IP] loaded: {len(attn2)} cross-attn layers, {num_tokens} tokens (encoder on {enc_device})")
    return {"image_proj": image_proj, "encoder": enc, "processor": proc,
            "enc_device": enc_device, "token_device": device, "dtype": dtype, "num_tokens": num_tokens}

def object_crop(ori_img, ori_mask, pad=0.10, gray_bg=True):
    m = (np.asarray(ori_mask) > 0)
    if m.ndim == 3: m = m[..., 0]
    ys, xs = np.where(m)
    if len(ys) == 0:
        return Image.fromarray(np.asarray(ori_img).astype(np.uint8))
    H, W = m.shape
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    py, px = int((y1 - y0) * pad), int((x1 - x0) * pad)
    y0, y1 = max(0, y0 - py), min(H, y1 + py + 1); x0, x1 = max(0, x0 - px), min(W, x1 + px + 1)
    img = np.asarray(ori_img).astype(np.uint8).copy()
    if gray_bg: img[~m] = 127
    return Image.fromarray(img[y0:y1, x0:x1])

@torch.no_grad()
def encode_tokens(handles, pil_crop):
    px = handles["processor"](images=pil_crop, return_tensors="pt").pixel_values
    px = px.to(handles["enc_device"], dtype=torch.float32)
    emb = handles["encoder"](px).image_embeds
    tok = handles["image_proj"](emb)
    return tok.to(handles["token_device"], dtype=handles["dtype"])

def ip_cross_branch(controller, attn_module, query_hb, local_region_flat):
    H = controller.heads
    tok = controller.ip_tokens.to(query_hb.dtype)
    ik = attn_module.ip_to_k(tok); iv = attn_module.ip_to_v(tok)
    ik = controller.head_to_batch_dim(ik); iv = controller.head_to_batch_dim(iv)
    def one(chunk_start):
        q = query_hb[chunk_start * H:(chunk_start + 1) * H]
        out = controller.mask_attention(q, ik, iv, None)
        out = controller.batch_to_head_dim(out)[0]
        return controller.ip_scale * local_region_flat[:, None] * out
    return one(0), one(2)

# ===== geometry-conditioning token builders (Approach 2 + HOG-IP) =====
def build_affine(edit_param, ori_mask, size=512):
    import cv2
    dx, dy, dz, rx, ry, rz, sx, sy, sz = edit_param
    m = np.asarray(ori_mask)
    if m.ndim == 3: m = m[..., 0]
    ys, xs = np.where(m > 0)
    if len(ys) == 0: cx, cy = size / 2.0, size / 2.0
    else: cx, cy = (xs.max() + xs.min()) / 2.0, (ys.max() + ys.min()) / 2.0
    M = cv2.getRotationMatrix2D((float(cx), float(cy)), -float(rz), 1.0)
    tx, ty = (1 - sx) * cx, (1 - sy) * cy
    M[0, 2] += dx + tx; M[1, 2] += dy + ty; M[0, 0] *= sx; M[1, 1] *= sy
    return M.astype(np.float32)

def warp_to_target(img, M, size=512):
    import cv2
    return cv2.warpAffine(np.asarray(img).astype(np.uint8), M, (size, size), flags=cv2.INTER_LANCZOS4)

def soft_edge(img, lineart=True):
    import cv2
    a = np.asarray(img)
    g = cv2.cvtColor(a, cv2.COLOR_RGB2GRAY) if a.ndim == 3 else a
    gx = cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3); gy = cv2.Sobel(g, cv2.CV_32F, 0, 1, ksize=3)
    mag = np.sqrt(gx * gx + gy * gy); mag = mag / (mag.max() + 1e-6) * 255.0
    if lineart: mag = 255.0 - mag
    e = np.clip(mag, 0, 255).astype(np.uint8)
    return Image.fromarray(np.stack([e, e, e], axis=-1))

@torch.no_grad()
def build_ip_tokens(handles, ori_img, ori_mask, target_mask, edit_param, mode="orig", geo_weight=1.0):
    if mode == "orig":
        return encode_tokens(handles, object_crop(ori_img, ori_mask))
    M = build_affine(edit_param, ori_mask); warped = warp_to_target(ori_img, M)
    if mode == "warp":
        return encode_tokens(handles, object_crop(warped, target_mask))
    if mode == "hogip":
        id_tok = encode_tokens(handles, object_crop(ori_img, ori_mask))
        geo_img = soft_edge(object_crop(warped, target_mask), lineart=True)
        geo_tok = encode_tokens(handles, geo_img) * float(geo_weight)
        return torch.cat([id_tok, geo_tok], dim=1)
    raise ValueError("unknown IP mode: " + str(mode))

In [ ]:
# ===== IP-B: attention.py IP hook + script (load + per-case build_ip_tokens) =====
A="/kaggle/temp/FreeFine/src/utils/attention.py"; a=open(A).read()
S="/kaggle/temp/FreeFine/evaluation/FreeFine/freefine_sweep_2d.py"; s=open(S).read()
def P(t,old,new,name):
    assert old in t, f"ANCHOR NOT FOUND: {name}"
    assert new not in t, f"ALREADY PATCHED: {name}"
    return t.replace(old,new,1)
a=P(a,
"                hidden_states = controller.modulate_local_cross_attn(query,key,value,is_cross, place_in_unet)",
"                hidden_states = controller.modulate_local_cross_attn(query,key,value,is_cross, place_in_unet, attn_module=self)","call-site")
a=P(a,
"    def modulate_local_cross_attn(self,query,key,value,is_cross, place_in_unet):",
"    def modulate_local_cross_attn(self,query,key,value,is_cross, place_in_unet, attn_module=None):","signature")
a=P(a,
"        _, L1, L2 = hidden_states.shape #4,4096,320 [uncon_edit,uncon_ref,con_edit,con_ref]\n        uncon_edit,uncon_ref,con_edit,_ = hidden_states\n",
"        _, L1, L2 = hidden_states.shape #4,4096,320 [uncon_edit,uncon_ref,con_edit,con_ref]\n        uncon_edit,uncon_ref,con_edit,_ = hidden_states\n"
"        if getattr(self,'ip_tokens',None) is not None and attn_module is not None and getattr(attn_module,'ip_to_k',None) is not None:\n"
"            import ip_ff as _ipff\n"
"            _au,_ac = _ipff.ip_cross_branch(self, attn_module, query, local_region)\n"
"            uncon_edit = uncon_edit + _au; con_edit = con_edit + _ac\n","ip-add")
open(A,"w").write(a); print("attention.py patched")
s=P(s,
"    model.enable_xformers_memory_efficient_attention()",
"    model.enable_xformers_memory_efficient_attention()\n"
"    if os.environ.get('FF_IP')=='1':\n"
"        import ip_ff\n"
"        from huggingface_hub import hf_hub_download\n"
"        _ckpt=hf_hub_download('h94/IP-Adapter','models/ip-adapter_sd15.bin')\n"
"        model._ip=ip_ff.load_ip(model,_ckpt,'h94/IP-Adapter',device)\n","ip-load")
s=P(s,
"        generated_results = model.FreeFine_generation(**params)",
"        if os.environ.get('FF_IP')=='1':\n"
"            import ip_ff\n"
"            model.controller.ip_tokens=ip_ff.build_ip_tokens(model._ip, ori_img, ori_mask, target_mask, case['edit_param'], mode=os.environ.get('FF_IP_MODE','orig'), geo_weight=float(os.environ.get('FF_GEO_WEIGHT','1.0')))\n"
"            model.controller.ip_scale=float(os.environ.get('FF_IP_SCALE','0.6'))\n"
"        else:\n"
"            model.controller.ip_tokens=None\n"
"        generated_results = model.FreeFine_generation(**params)","ip-percase")
open(S,"w").write(s); print("freefine_sweep_2d.py patched")

In [ ]:
# ===== C8 generate =====
import os, time, glob, subprocess, socket, numpy as np
from PIL import Image
GEO="/kaggle/temp/GeoBenchMeta"; OUT="/kaggle/working/exp3ip/variants"; os.makedirs(OUT,exist_ok=True)
GENBASE=glob.glob("/kaggle/input/**/gen_results_2d_final/gen_results_2d_backup",recursive=True)[0]
GEN_DEADLINE=NB_START+6.5*3600
NEW=[("ip_orig",  {"FF_IP":1,"FF_IP_MODE":"orig","FF_IP_SCALE":0.6}),
     ("warp",     {"FF_IP":1,"FF_IP_MODE":"warp","FF_IP_SCALE":0.6}),
     ("hogip_g05",{"FF_IP":1,"FF_IP_MODE":"hogip","FF_IP_SCALE":0.6,"FF_GEO_WEIGHT":0.5}),
     ("hogip_g10",{"FF_IP":1,"FF_IP_MODE":"hogip","FF_IP_SCALE":0.6,"FF_GEO_WEIGHT":1.0})]
def gen2(o,**kw):
    os.makedirs(o,exist_ok=True); env=os.environ.copy()
    env.update({"PATH":"/kaggle/temp/freefine_env/bin:"+env["PATH"],"PYTHONUNBUFFERED":"1","TOKENIZERS_PARALLELISM":"false","HF_HOME":"/kaggle/temp/hf","NCCL_P2P_DISABLE":"1","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True","FF_SUBSET_JSON":f"{GEO}/gen_subset.json","FF_OUT_DIR":o})
    env.update({k:str(v) for k,v in kw.items()})
    s=socket.socket();s.bind(("",0));p=s.getsockname()[1];s.close()
    return subprocess.run(["/kaggle/temp/freefine_env/bin/torchrun","--nproc_per_node=2","--master-port",str(p),"freefine_sweep_2d.py"],cwd="/kaggle/temp/FreeFine/evaluation/FreeFine",env=env,stdout=open(o+"/log.txt","w"),stderr=subprocess.STDOUT)
def pdiff(tag):
    ds=[]
    for n in sorted(glob.glob(f"{OUT}/{tag}/**/*.png",recursive=True))[:8]:
        rel=os.path.relpath(n,f"{OUT}/{tag}")
        ds.append(round(float(np.abs(np.array(Image.open(n).convert("RGB"),float)-np.array(Image.open(f"{GENBASE}/{rel}").convert("RGB").resize(Image.open(n).size),float)).mean()),2))
    return ds
done=[]
for tag,kw in NEW:
    if time.time()>GEN_DEADLINE: print("GEN_DEADLINE skip",tag,flush=True); break
    o=f"{OUT}/{tag}"; t=time.time()
    if len(glob.glob(o+"/**/*.png",recursive=True))>=200:
        print(f"skip {tag}: already generated",flush=True); done.append(tag); continue
    try:
        r=gen2(o,**kw); k=len(glob.glob(o+"/**/*.png",recursive=True))
        print(f"[{int((time.time()-NB_START)/60)}m] {tag}: rc={r.returncode} {int(time.time()-t)}s imgs={k} Δvs_baseline={pdiff(tag)}",flush=True)
        if k>0: done.append(tag)
        if r.returncode!=0: print("  tail:",open(o+'/log.txt').read()[-1200:])
    except Exception as ex: print(f"{tag} FAILED:{ex}",flush=True)
print("new variants:",done,flush=True)

In [ ]:
# ===== C9 eval (Account 1: IP-pathway arms + coarse ceiling row) =====
import os, json, re, glob, subprocess, time, numpy as np
from PIL import Image
GEO="/kaggle/temp/GeoBenchMeta"; MET="/kaggle/temp/FreeFine/evaluation/metrics"; PY="/kaggle/temp/metric_env/bin/python"
EVAL_DEADLINE=NB_START+10.5*3600
ann=json.load(open(f"{GEO}/annotation_2d.json")); picked=json.load(open(f"{GEO}/subset_meta.json"))
for tag in ["ip_orig","warp","hogip_g05","hogip_g10"]:
    p=f"/kaggle/working/exp3ip/variants/{tag}"
    if glob.glob(p+"/**/*.png",recursive=True):
        d=f"{GEO}/gen_eval/{tag}"
        if os.path.islink(d): os.remove(d)
        os.symlink(p,d)
# coarse = eval-only ceiling/floor row (no generation; the perfect in-plane warp itself)
d=f"{GEO}/gen_eval/coarse"
if os.path.islink(d): os.remove(d)
os.symlink(f"{GEO}/Geo-Bench-2D/coarse_img", d)
sets=[s for s in ["baseline","coarse","ip_orig","warp","hogip_g05","hogip_g10"] if os.path.exists(f"{GEO}/gen_eval/{s}")]
print("eval sets:",sets)
def mem(pred): return [(r["da_n"],r["ins_id"],r["case_id"]) for r in picked if pred(r)]
groups={"rotate_hard":(mem(lambda r:r["edit_type"]=="rotate" and r["difficulty"]=="hard"),"000110100"),
        "resize_hard":(mem(lambda r:r["edit_type"]=="resize" and r["difficulty"]=="hard"),"000110100"),
        "move_all":(mem(lambda r:r["edit_type"]=="move"),"000110000"),
        "all_200":(mem(lambda r:True),"100110011")}
def wrap_e(ids,gd):
    tot=0.0;n=0
    for d,i,e in ids:
        cp=f"{GEO}/Geo-Bench-2D/coarse_img/{d}/{i}/{e}.png"; gp=f"{gd}/{d}/{i}/{e}.png"; tp=f"{GEO}/Geo-Bench-2D/target_mask/{d}/{i}/{e}.png"
        if not(os.path.exists(cp) and os.path.exists(gp) and os.path.exists(tp)): continue
        C=np.array(Image.open(cp).convert("RGB"),float)/255; G=np.array(Image.open(gp).convert("RGB"),float)/255; T=np.array(Image.open(tp).convert("L"),float)/255
        if G.shape[:2]!=C.shape[:2]: G=np.array(Image.fromarray((G*255).astype("uint8")).resize((C.shape[1],C.shape[0])),float)/255
        if T.shape[:2]!=C.shape[:2]: T=np.array(Image.fromarray((T*255).astype("uint8")).resize((C.shape[1],C.shape[0])),float)/255
        mm=np.repeat(T[...,None],3,axis=2); su=mm.sum()
        if su<=0: continue
        tot+=float(np.sum(np.abs(C*mm-G*mm))/su); n+=1
    return round(tot/n,4) if n else None
def manifest(setn,ids):
    o={}; b=f"{GEO}/gen_eval/{setn}"
    for d,i,e in ids:
        if not os.path.exists(f"{b}/{d}/{i}/{e}.png"): continue
        lf=dict(ann[d]["instances"][i][e]); lf["gen_img_path"]=f"gen_eval/{setn}/{d}/{i}/{e}.png"
        o.setdefault(d,{"instances":{}})["instances"].setdefault(i,{})[e]=lf
    pth=f"{GEO}/m_{setn}_{len(ids)}.json"; json.dump(o,open(pth,"w")); return pth
def runm(manp,task):
    env=os.environ.copy(); env.update({"MPLBACKEND":"Agg","HF_HOME":"/kaggle/temp/hf","TORCH_HOME":"/kaggle/temp/torch","PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True"})
    out=subprocess.run([PY,"main.py","--path",manp,"--use_relative_path","--base_dir",GEO,"--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2","--task",task,"--level","0"],cwd=MET,env=env,capture_output=True,text=True)
    t=out.stdout+out.stderr; v={}
    for k in ["FID_DINO","FID_KD","FID","SUBC","BGC","MD"]:
        mr=re.findall(rf"(?:^|\s){k}:\s*([-\d.eE]+)",t)
        if mr: v[k]=round(float(mr[-1]),4)
    return v
results={}; stop=False
for setn in sets:
    if stop: break
    results[setn]={}
    for g,(ids,task) in groups.items():
        if time.time()>EVAL_DEADLINE: print("EVAL_DEADLINE",flush=True); stop=True; break
        if not ids: continue
        v=runm(manifest(setn,ids),task); v["WRAP_E"]=wrap_e(ids,f"{GEO}/gen_eval/{setn}"); v["n"]=len(ids)
        results[setn][g]=v; print(f"{setn:12s} {g:12s} -> {v}",flush=True)
    json.dump(results,open("/kaggle/working/exp3ip_full.json","w"),indent=2)
print("eval done")

In [ ]:
# ===== C10 summary (Account 1) =====
import json
R=json.load(open("/kaggle/working/exp3ip_full.json"))
order=["baseline","coarse","ip_orig","warp","hogip_g05","hogip_g10"]
for g in ["rotate_hard","resize_hard","move_all","all_200"]:
    print(f"\n=== {g} ===")
    cols=["SUBC","BGC","WRAP_E","MD","FID","FID_DINO","FID_KD"]
    print("set          "+"".join(f"{c:>10}" for c in cols))
    for s in order:
        if s in R and g in R[s]:
            d=R[s][g]; print(f"{s:12s} "+"".join(f"{str(d.get(c,'-')):>10}" for c in cols))
print("\nEXP-3 IP-pathway: ip_orig = B2 reference (source crop); warp = Approach 2 (crop warped to target pose);")
print("hogip_g05/g10 = HOG-IP (identity tokens + soft-edge geometry tokens at weight 0.5/1.0).")
print("coarse = the perfect in-plane warp evaluated as-is (ceiling/floor row; its WRAP_E is 0 by definition).")
print("GATE: MD ↓ >= 1.0 on hard groups + WRAP_E ↓, with SUBC/BGC not regressing. SUBC is reported, not gated (proven ceiling).")
print("saved -> /kaggle/working/exp3ip_full.json")